In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, f_oneway
import logging
from diffprivlib.models import GaussianNB  # Simulated differential privacy
from joblib import dump
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

In [6]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [9]:
df = pd.read_csv('Viral_Social_Media_Trends.csv')

In [10]:
# Privacy-awareData Loading
def load_data_privacy_aware(file_path):
    logging.info("Loading data with privacy simulation")
    df = pd.read_csv(file_path)
    # Simulates differential privacy noise on numeric columns
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    for col in numeric_cols:
        noise = np.random.laplace(0, 0.1, size=len(df))  # Small noise for demo
        df[col] = df[col] + noise
        df[col] = df[col].clip(lower=0)  # Ensuring non-negatives
    return df

In [ ]:
# Dropping missing values
df.dropna(inplace=True)

# Converting categorical columns to categorical type
for col in ['Platform', 'Hashtag', 'Content_Type', 'Region', 'Engagement_Level']:
    df[col] = df[col].astype('category')

# Creating a new feature based on existing data
df['Log_Views'] = np.log1p(df['Views'].clip(lower=0))

In [ ]:
# Grouping data to find the most engaging hashtag per platform 
top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
df['Top_Hashtag_Platform'] = df['Platform'].map(top_hashtags)

In [14]:
# Visualization Generation (Minimalist Design)
def generate_visualizations(df):
    logging.info("Generating visualizations with Apple-inspired design")
    sns.set_style("white")  # Clean, minimalist look
    plt.rcParams['font.family'] = 'Helvetica'
    # Bar Chart
    platform_eng = df.groupby('Platform').agg({'Engagement_Score': 'mean'}).reset_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=platform_eng, x='Platform', y='Engagement_Score', palette='colorblind')
    plt.title("Engagement by Platform", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")  # Minimal labels
    plt.savefig("engagement_by_platform.png", dpi=300, bbox_inches='tight')
    plt.close() 
    # Pie Chart
    content_dist = df['Content_Type'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(content_dist, labels=content_dist.index, autopct='%1.1f%%', colors=sns.color_palette('colorblind', n_colors=len(content_dist)), 
            textprops={'fontsize': 12})
    plt.title("Content Type Distribution", fontsize=16, pad=20)
    plt.savefig("content_type_distribution.png", dpi=300, bbox_inches='tight')
    plt.close()
    # Grouped Bar Chart
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='Engagement_Level', y='Hashtag_Sentiment', hue='Platform', palette='colorblind')
    plt.title("Sentiment by Engagement", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")
    plt.legend().remove()  # Minimalist: remove legend if context clear
    plt.savefig("sentiment_by_engagement.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Line Chart
    df['Index'] = range(len(df))
    plt.figure(figsize=(12, 6))
    plt.plot(df['Index'], df['Engagement_Score'], color=sns.color_palette('colorblind')[0], label='Actual')
    plt.plot(df['Index'], df['Trend_Prediction'], color=sns.color_palette('colorblind')[1], label='Trend')
    plt.title("Engagement Trend", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")
    plt.legend(frameon=False)
    plt.savefig("engagement_trend.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Heatmap
    plt.figure(figsize=(10, 6))
    pivot = df.pivot_table(values='Engagement_Score', index='Region', columns='Cluster_Name', aggfunc='mean')
    sns.heatmap(pivot, cmap='Blues', annot=True, fmt='.2f', cbar=False, annot_kws={"size": 12})
    plt.title("Engagement by Region & Cluster", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")
    plt.savefig("engagement_heatmap.png", dpi=300, bbox_inches='tight')
    plt.close()

In [28]:
# Save Processed Data
df.to_pickle("processed_data_apple.pkl")
logging.info("EDA core completed and data saved")

2025-03-18 09:08:48,050 - INFO - EDA core completed and data saved
